In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mping
import random

import os

# Visualisando e manipulando dados

In [2]:
CAMINHO = "C:\\Users\\Nitro\\Desktop\\Deep_Learning\\ID para o jhonny-20260515T045215Z-3-001\\ID para o jhonny"

In [18]:
diretorios = os.listdir(CAMINHO)

# Dicionário chave: diretório, valor: lista de imagens
arquivos = {dir : os.listdir(CAMINHO + "\\" + dir) for dir in diretorios if dir[-4] != "."}
#print(arquivos[list(arquivos.keys())[0]])
# Lista de familias para criar datasets no estilo do tensorflow
lista_familias = list(arquivos.keys())
#print(lista_familias)
arquivos = {dir: arquivos[dir] for dir in arquivos.keys() if len(arquivos[dir]) > 20}
arquivos.keys()




dict_keys(['Arecaceae', 'Bignoniaceae', 'Euphorbiaceae', 'Fabaceae', 'Lauraceae', 'Meliaceae', 'Myrtaceae'])

# Verificando quantidade de imagens por classe

# Criando caminhos no estilo do tensor flow
# FEITO

In [15]:
caminho_test ="C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\test"
caminho_treino = "C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\train"
caminho_validation = "C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\val"

In [16]:
def construir_caminhos(test, train, val):
    for familia in lista_familias:
        os.makedirs(test + "\\" + familia, exist_ok=True)
        os.makedirs(train + "\\" + familia, exist_ok=True)
        os.makedirs(val + "\\" + familia, exist_ok=True)
        print("Pastas de famílias criadas com sucesso!!!")
#construir_caminhos(caminho_test, caminho_treino, caminho_validation)
# Pastas de famílias foram criadas com sucesso

# Criando função que copia fotos
# FEITO

In [20]:
def copy_image(caminho_origem: str, percents, caminho_destino_test: str, caminho_destino_train:str, caminho_destino_val:str):
    import random
    import shutil



    # Variável "arquivos"
    for familia in arquivos.keys():
        print(familia)
        imagens_treino = random.sample(arquivos[familia], int(len(arquivos[familia]) * percents["train"]))
        sobra_treino = [x for x in arquivos[familia] if x not in imagens_treino]
        imagens_teste = random.sample(sobra_treino, int(len(sobra_treino) * 0.66))
        imagens_val = [x for x in sobra_treino if x not in imagens_teste]

        for foto in imagens_treino:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_treino = caminho_destino_train + "\\" + familia
            shutil.copy2(caminho_completo_origem, caminho_completo_destino_treino)


        for foto in imagens_teste:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_teste = caminho_destino_test + "\\" + familia

            shutil.copy2(caminho_completo_origem, caminho_completo_destino_teste)

        for foto in imagens_val:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_val = caminho_destino_val + "\\" + familia

            shutil.copy2(caminho_completo_origem, caminho_completo_destino_val)




    return
#copy_image(CAMINHO, {"test": 0.2, "train": 0.7, "val": 0.1}, caminho_test, caminho_treino, caminho_validation)

Arecaceae
Bignoniaceae
Euphorbiaceae
Fabaceae
Lauraceae
Meliaceae
Myrtaceae


# Carregar as imagens para a rede neural

In [21]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [26]:
from tensorflow.keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(caminho_treino, image_size=(192, 192), batch_size=32)
val_dataset = image_dataset_from_directory(caminho_validation, image_size=(192, 192), batch_size=32)
test_dataset = image_dataset_from_directory(caminho_test, image_size=(192,192), batch_size=32)

Found 190 files belonging to 7 classes.
Found 33 files belonging to 7 classes.
Found 51 files belonging to 7 classes.


In [27]:
for data_batch, labels_batch in train_dataset:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    print(data_batch[0].shape)
    break

data batch shape: (32, 192, 192, 3)
labels batch shape: (32,)
(192, 192, 3)


# Treinando o Modelo

In [28]:
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base_model = MobileNetV2(input_shape=(192, 192, 3),
                         include_top=False,
                         weights="imagenet")

base_model.trainable = False

model = keras.Sequential([
    keras.Input(shape=(192, 192, 3)),

    Rescaling(1./255),
    RandomFlip('horizontal'),
    RandomRotation(0.1),
    RandomZoom(0.2),

    keras.layers.Lambda(preprocess_input),

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
#model.fit(train_dataset, epochs=20)